In [ ]:
import os
import pathlib
import logging
import tarfile

import numina.util.context as ctx
from numina.user.session import Session
from numina.testing.testcache import download_cache
from megaradrp.tools.init_tree import ensure_numina_cfg, init_tree, install_calibration_data


import matplotlib.pyplot as plt

In [ ]:
logging.basicConfig(level=logging.DEBUG)

In [ ]:
workdir = pathlib.Path().resolve()

In [ ]:
dataset = 'megara-cookbook-v4'
url = 'https://guaix.fis.ucm.es/data/megaradrp/{}.tar.gz'.format(dataset)

downloaded = download_cache(url)

# Uncompress
with tarfile.open(downloaded.name, mode="r:gz") as tar:
    tar.extractall(filter="data")

os.remove(downloaded.name)

In [ ]:
basedir = workdir / dataset / 'M15_LCB_HR-R'
# .numina.cfg and the calibration tree are in the directory of M15, as with megaradrp-inittree
os.chdir(basedir)
logger = logging.getLogger('inittree')
calibrations_url = 'https://zenodo.org/records/18623771/files/guaix-ucm/megaradrp-calibrations-2026.2.zip'
init_tree(base_path='calibrations', logger=logger)
install_calibration_data(url=calibrations_url, base_path='calibrations', overwrite=False, logger=logger)
ensure_numina_cfg(logger=logger)

# Registry of the reductions, the products are found there in later reductions
session = Session(basedir=basedir, db="numina-db.json")

In [ ]:
obsresults = ["0_Bias.yaml", "1_TraceMap.yaml",
         "3_WaveCalib.yaml", "4_FiberFlat.yaml",
         "5_TwilightFlat.yaml", "6_LcbAdquisition.yaml",
         "7_StandardStar.yaml", "8_LcbImage.yaml"]

session.add_observations(*obsresults)

In [ ]:
obsid = "0_Bias"
task0 = session.run(obsid)

In [ ]:
task0.result.qc

In [ ]:
master_bias = task0.result.master_bias.open()

In [ ]:
master_bias[0].header

In [ ]:
task0.request_runinfo

In [ ]:
obsid = "1_TraceMap_HR-R"
task1 = session.run(obsid)

In [ ]:
print(task1.result.master_traces)

In [ ]:
task1.result.master_traces

In [ ]:
obsid = "3_WaveCalib_HR-R"
task3 = session.run(obsid)

In [ ]:
obsid = "4_FiberFlat_HR-R"
task4 = session.run(obsid)

In [ ]:
obsid = "5_TwilightFlat_HR-R"
task5 = session.run(obsid)

In [ ]:
obsid = "6_LcbImage_HR-R"
task6 = session.run(obsid)

In [ ]:
obsid = "7_StandardStar_HR-R"
task7 = session.run(obsid)

In [ ]:
obsid = "8_LcbImage_HR-R_M15"
task8 = session.run(obsid)

In [ ]:
rss = task8.result.reduced_rss.open()

In [ ]:
plt.imshow(rss[0].data)

In [ ]:

from astropy.wcs import WCS
import numpy as np

In [ ]:
projection = WCS(rss['FIBERS'].header)

In [ ]:
def hexplot_helper(ax, img):
    import megaradrp.visualization as vis
    import megaradrp.datamodel as dm
    # This should be simplified
    rssdata = np.squeeze(img[0].data[:,2000])
    datamodel = dm.MegaraDataModel()
    fiberconf = datamodel.get_fiberconf(rss)
    x = np.empty((fiberconf.nfibers,))
    y = np.empty((fiberconf.nfibers,))
    # Key is fibid
    for _, fiber in sorted(fiberconf.fibers.items()):
        idx = fiber.fibid - 1
        x[idx] = fiber.x
        y[idx] = fiber.y
    scale = 0.443
    col = vis.hexplot(ax, x,y, rssdata, scale=scale)
    return col
    

In [ ]:
fig = plt.figure()
ax = fig.add_axes([0.15, 0.1, 0.8, 0.8], projection=projection)

ax.coords.grid()
ax.set_xlim([-6.5, 6.5])
ax.set_ylim([-6, 6])
col = hexplot_helper(ax, rss)